In [0]:
# 1. SETUP DE AMBIENTE E GOVERNANÇA DO SCHEMA GOLD

# Mesmo racional das células de setup anteriores — mantido por consistência
# entre os 3 notebooks do pipeline.

from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "workspace"
silver_schema_name = "silver"
gold_schema_name = "gold"

silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

# Provisionamento formal do schema Gold
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

print(f"Origem Silver: {silver_schema}")
print(f"Destino Gold:  {gold_schema}")

Origem Silver: workspace.silver
Destino Gold:  workspace.gold


In [0]:
# 2. CONSTRUÇÃO DAS DIMENSÕES DESCRITIVAS (STAR SCHEMA COM SURROGATE KEYS)

# 2.1. gold.dim_movies (Dimensão Central de Filmes)
# DECISÃO: `row_number()` ordenado por `id_filme` para gerar `sk_movie_id`,
# em vez de `monotonically_increasing_id()` (sugerido na docmentação do projeto). Preferimos `row_number()` porque ele gera SKs sequenciais e
# reprodutíveis (mesma ordenação → mesmos IDs em reprocessamentos), o que
# facilita debug e comparação entre execuções. `monotonically_increasing_id()`
# é mais rápido em datasets distribuídos muito grandes (não exige um único
# shuffle ordenado), mas gera IDs não-determinísticos entre execuções — um
# trade-off performance vs. reprodutibilidade que decidimos resolver a favor
# da reprodutibilidade, dado o volume do dataset (~200k linhas) não justificar
# o ganho de performance
# ------------------------
df_silver_info = spark.table(f"{silver_schema}.tb_info_filmes")

janela_sk_movie = Window.orderBy("id_filme")

df_dim_movies = (
    df_silver_info
    .withColumn("sk_movie_id", F.row_number().over(janela_sk_movie).cast("bigint"))
    .select(
        "sk_movie_id",
        "id_filme",
        "titulo",
        "data_lancamento",
        "ano_lancamento",
        "duracao_minutos",
        "idioma_original",
        "status_filme",
        "sinopse"
    )
)

(
    df_dim_movies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.dim_movies")
)
print(f"✔ Tabela {gold_schema}.dim_movies gravada ({df_dim_movies.count()} filmes).")

# 2.2. gold.dim_genres (Catálogo de Gêneros)
# DECISÃO: catálogo deduplicado via `.distinct()` antes de gerar a SK, para
# garantir que cada gênero apareça só uma vez no catálogo — pré-requisito
# para a bridge table funcionar sem duplicar o grão da fato mais adiante.
# ------------------------------------------------------------------------------
df_silver_generos = spark.table(f"{silver_schema}.tb_generos")

janela_sk_genre = Window.orderBy("nome_genero")

df_dim_genres = (
    df_silver_generos
    .select("nome_genero")
    .distinct()
    .filter(F.col("nome_genero").isNotNull() & (F.col("nome_genero") != ""))
    .withColumn("sk_genre_id", F.row_number().over(janela_sk_genre).cast("bigint"))
    .select("sk_genre_id", "nome_genero")
)

(
    df_dim_genres.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.dim_genres")
)
print(f"✔ Tabela {gold_schema}.dim_genres gravada ({df_dim_genres.count()} gêneros únicos).")


# 2.3. gold.dim_companies (Catálogo de Produtoras)
# DECISÃO: filtramos `tipo_entidade == "Produtora"` a partir da tabela
# unificada da Silver, em vez de ter uma tabela Silver separada só para
# produtoras. Isso mantém a "dimensão unificada" pedida na Silver (Seção
# 1.3, item 6) e delega à Gold a responsabilidade de separar por tipo de
# negócio (pessoas físicas vs. empresas) — que é onde essa distinção importa
# para o modelo dimensional (dim_people vs. dim_companies).
# ------------------------------------------------------------------------------
df_silver_entidades = spark.table(f"{silver_schema}.tb_pessoas_empresas")

janela_sk_company = Window.orderBy("nome_produtora")

df_dim_companies = (
    df_silver_entidades
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.col("nome_entidade").alias("nome_produtora"))
    .distinct()
    .filter(F.col("nome_produtora").isNotNull() & (F.col("nome_produtora") != ""))
    .withColumn("sk_company_id", F.row_number().over(janela_sk_company).cast("bigint"))
    .select("sk_company_id", "nome_produtora")
)

(
    df_dim_companies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.dim_companies")
)
print(f"✔ Tabela {gold_schema}.dim_companies gravada ({df_dim_companies.count()} produtoras únicas).")

# 2.4. gold.dim_people (Catálogo de Profissionais do Audiovisual)
# DECISÃO: grão da SK é (`nome_pessoa`, `tipo_pessoa`), não só `nome_pessoa`.
# Isso é proposital: a mesma pessoa pode aparecer como Ator em um filme e
# Diretor em outro (comum no cinema), e o projeto define `dim_people` como
# tendo `tipo_pessoa` como atributo — então tratamos essas combinações como
# entidades distintas no catálogo, refletindo o "papel" e não só a "pessoa".
# Uma modelagem alternativa (uma linha por pessoa, um `tipo_pessoa` array ou
# uma bridge pessoa↔papel) captaria melhor "a mesma pessoa" como entidade
# única, mas o projeto pede `tipo_pessoa` como coluna simples de
# `dim_people`, então seguimos a especificação literal
# ------------------------------------------------------------------------------

janela_sk_person = Window.orderBy("nome_pessoa", "tipo_pessoa")

df_dim_people = (
    df_silver_entidades
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(
        F.col("nome_entidade").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa")
    )
    .distinct()
    .filter(F.col("nome_pessoa").isNotNull() & (F.col("nome_pessoa") != ""))
    .withColumn("sk_person_id", F.row_number().over(janela_sk_person).cast("bigint"))
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
)

(
    df_dim_people.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.dim_people")
)
print(f"✔ Tabela {gold_schema}.dim_people gravada ({df_dim_people.count()} profissionais únicos).")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✔ Tabela workspace.gold.dim_movies gravada (97611 filmes).
✔ Tabela workspace.gold.dim_genres gravada (19 gêneros únicos).
✔ Tabela workspace.gold.dim_companies gravada (46538 produtoras únicas).
✔ Tabela workspace.gold.dim_people gravada (420012 profissionais únicos).


In [0]:

# 3. GOLD: DIM_REVIEWS (MÉTRICAS RESUMIDAS DE USUÁRIOS POR FILME)

# DECISÃO: agregamos as avaliações ANTES de conectar à fato, gerando uma
# tabela de métricas resumidas (contagem + média) em vez de deixar a
# granularidade de avaliação-por-usuário na Gold. O projeto pede
# explicitamente "transformando-as em uma métrica resumida por filme" — a
# decisão de negócio aqui é: o time de BI/consumo da Gold não precisa
# analisar avaliação individual, só o comportamento agregado por filme.
# `inner join` com `dim_movies` (não `left`): filmes sem nenhuma avaliação
# simplesmente não aparecem em `dim_reviews`, o que é semanticamente
# correto — "zero avaliações" e "não tem linha nesta dimensão" comunicam a
# mesma informação sem precisar de uma linha com contagem=0



df_silver_reviews = spark.table(f"{silver_schema}.tb_avaliacoes_usuarios")
lookup_dim_movies = spark.table(f"{gold_schema}.dim_movies").select("id_filme", "sk_movie_id")

# 1. Agregação das notas válidas por ID do filme
df_reviews_resumo = (
    df_silver_reviews
    .filter(F.col("nota_usuario").isNotNull())
    .groupBy("id_filme")
    .agg(
        F.count("nota_usuario").cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios")
    )
)

# 2. Associação com sk_movie_id e atribuição da SK própria
janela_sk_rev = Window.orderBy("sk_movie_id")

df_dim_reviews = (
    df_reviews_resumo
    .join(lookup_dim_movies, on="id_filme", how="inner")
    .withColumn("sk_review_id", F.row_number().over(janela_sk_rev).cast("bigint"))
    .select(
        "sk_review_id",
        "sk_movie_id",
        "qtd_avaliacoes_usuarios",
        "nota_media_usuarios"
    )
)

(
    df_dim_reviews.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.dim_reviews")
)

print(f" Tabela {gold_schema}.dim_reviews gravada ({df_dim_reviews.count()} filmes com avaliações).")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


 Tabela workspace.gold.dim_reviews gravada (26038 filmes com avaliações).


In [0]:

# 4. TABELAS-PONTE (BRIDGE TABLES) PARA RELACIONAMENTOS N:N
# DECISÃO CENTRAL (vale para as 3 bridges): uma FK direta de
# `fact_movies_performance` para gêneros/pessoas/produtoras quebraria o
# grão "um registro por filme" assim que um filme tivesse mais de uma
# associação (a maioria tem vários gêneros e vários atores). Isso causaria
# "Cartesian Explosion": ao dar join da fato com a lista de atores de um
# filme, a linha do filme se multiplicaria por N atores, e qualquer soma de
# receita/orçamento passaria a contar o mesmo valor N vezes. A bridge table
# resolve isso desacoplando as dimensões periféricas do núcleo da fato — o
# padrão Kimball citado no enunciado.

# DECISÃO 2: (inner join nos lookups): usamos `inner join` (não `left`) entre
# Silver e as dimensões — uma associação só entra na bridge se tanto o filme
# quanto a entidade (gênero/pessoa/produtora) já existirem nas respectivas
# dimensões. Isso previne "chaves órfãs" na bridge (FKs apontando para SKs
# inexistentes), garantindo integridade referencial sem precisar de
# constraints explícitas do banco (que o Delta Lake não impõe por padrão).
#
# DECISÃO 3: (`.distinct()` no fim de cada bridge): a Silver pode ter, em
# teoria, o mesmo par filme-entidade vindo de fontes diferentes ou de
# reprocessamento; `.distinct()` garante que a bridge não infle a
# cardinalidade N:N artificialmente com pares repetidos.


# Tabelas dimensionais carregadas para lookup de chaves
df_lookup_movies = spark.table(f"{gold_schema}.dim_movies").select("id_filme", "sk_movie_id")
df_lookup_genres = spark.table(f"{gold_schema}.dim_genres")
df_lookup_companies = spark.table(f"{gold_schema}.dim_companies")
df_lookup_people = spark.table(f"{gold_schema}.dim_people")

# ------------------------------------------------------------------------------
# 4.1. gold.bridge_movie_genre (Filme <-> Gênero)
# ------------------------------------------------------------------------------
df_bridge_genre = (
    spark.table(f"{silver_schema}.tb_generos")
    .join(df_lookup_movies, on="id_filme", how="inner")
    .join(df_lookup_genres, on="nome_genero", how="inner")
    .select(
        F.col("sk_movie_id").cast("bigint"),
        F.col("sk_genre_id").cast("bigint")
    )
    .distinct()
)

(
    df_bridge_genre.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.bridge_movie_genre")
)
print(f"✔ Tabela {gold_schema}.bridge_movie_genre gravada ({df_bridge_genre.count()} associações).")

# ------------------------------------------------------------------------------
# 4.2. gold.bridge_movie_person (Filme <-> Pessoa Física: Ator, Diretor, Roteirista)
# ------------------------------------------------------------------------------
df_silver_peop = spark.table(f"{silver_schema}.tb_pessoas_empresas").filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))

df_bridge_person = (
    df_silver_peop
    .join(df_lookup_movies, on="id_filme", how="inner")
    .join(
        df_lookup_people,
        (df_silver_peop.nome_entidade == df_lookup_people.nome_pessoa) &
        (df_silver_peop.tipo_entidade == df_lookup_people.tipo_pessoa),
        how="inner"
    )
    .select(
        F.col("sk_movie_id").cast("bigint"),
        F.col("sk_person_id").cast("bigint")
    )
    .distinct()
)

(
    df_bridge_person.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.bridge_movie_person")
)
print(f"✔ Tabela {gold_schema}.bridge_movie_person gravada ({df_bridge_person.count()} associações).")

# ------------------------------------------------------------------------------
# 4.3. gold.bridge_movie_company (Filme <-> Produtora)
# ------------------------------------------------------------------------------
df_silver_comp = spark.table(f"{silver_schema}.tb_pessoas_empresas").filter(F.col("tipo_entidade") == "Produtora")

df_bridge_company = (
    df_silver_comp
    .join(df_lookup_movies, on="id_filme", how="inner")
    .join(df_lookup_companies, df_silver_comp.nome_entidade == df_lookup_companies.nome_produtora, how="inner")
    .select(
        F.col("sk_movie_id").cast("bigint"),
        F.col("sk_company_id").cast("bigint")
    )
    .distinct()
)

(
    df_bridge_company.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.bridge_movie_company")
)
print(f"✔ Tabela {gold_schema}.bridge_movie_company gravada ({df_bridge_company.count()} associações).")

✔ Tabela workspace.gold.bridge_movie_genre gravada (139862 associações).
✔ Tabela workspace.gold.bridge_movie_person gravada (782048 associações).
✔ Tabela workspace.gold.bridge_movie_company gravada (119551 associações).


In [0]:
# 5. GOLD: FACT_MOVIES_PERFORMANCE (NÚCLEO DO STAR SCHEMA)

# DECISÃO 1 (filtro `status_filme == "Lançado"`): a documentação do projeto aponta: "a tabela
# fato deve consolidar métricas de filmes lançados sem duplicar grão através
# dos joins". Interpretamos isso como um filtro de negócio explícito, não
# implícito. Métricas de bilheteria/engajamento de filmes "Planejado" ou
# "Em Produção" não fazem sentido de negócio (não têm receita realizada
# ainda), então excluí-los da fato evita que análises de BI (ex.: receita
# total, rankings) sejam poluídas por filmes que sequer estrearam

# DECISÃO 2 (`left join` com financeiro/métricas): usamos `left` a partir de
# `df_filmes_elegiveis`, não `inner`. Isso é deliberado: um filme lançado
# sem dado financeiro disponível na origem ainda deve aparecer na fato (com
# métricas NULL), porque ele existe e foi lançado — excluí-lo (via inner
# join) enviesaria para baixo qualquer contagem de "quantos filmes
# lançados existem", mesmo que os valores financeiros de fato estejam
# ausentes

# 1. Base elegível da fato: exclusivamente filmes com status 'Lançado'
df_filmes_elegiveis = (
    spark.table(f"{gold_schema}.dim_movies")
    .filter(F.col("status_filme") == "Lançado")
    .select("sk_movie_id", "id_filme")
)

# 2. Leitura das fontes Silver pré-higienizadas
df_silver_fin = spark.table(f"{silver_schema}.tb_financeiro_filmes")
df_silver_met = spark.table(f"{silver_schema}.tb_metricas_engajamento")

# 3. Join 1:1 sem duplicação de grão
df_fact_movies = (
    df_filmes_elegiveis
    .join(df_silver_fin, on="id_filme", how="left")
    .join(df_silver_met, on="id_filme", how="left")
    .select(
        F.col("sk_movie_id").cast("bigint"),
        F.col("orcamento_usd").cast("decimal(18,2)"),
        F.col("receita_usd").cast("decimal(18,2)"),
        F.col("lucro_usd").cast("decimal(18,2)"),
        F.col("orcamento_brl").cast("decimal(18,2)"),
        F.col("receita_brl").cast("decimal(18,2)"),
        F.col("lucro_brl").cast("decimal(18,2)"),
        F.col("popularidade").cast("double"),
        F.col("nota_media_tmdb").cast("double"),
        F.col("qtd_votos_tmdb").cast("int"),
        F.col("nota_media_imdb").cast("double"),
        F.col("qtd_votos_imdb").cast("int")
    )
)

(
    df_fact_movies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.fact_movies_performance")
)

print(f" Tabela {gold_schema}.fact_movies_performance gravada ({df_fact_movies.count()} filmes lançados).")

 Tabela workspace.gold.fact_movies_performance gravada (96261 filmes lançados).


In [0]:
# 6. ENTREGA 2: TABELA DE CONTEXTO PARA LLM / RAG (VECTOR SEARCH)

# DECISÃO CENTRAL (a "casca de banana"): resolvemos cada campo
# potencialmente nulo em uma coluna intermediária própria (`v_titulo`,
# `v_ano`, `v_receita` etc.) ANTES de concatenar, em vez de tentar um único
# `coalesce()` gigante na expressão final. Isso torna cada fallback
# individualmente testável/legível e evita o comportamento de `concat()`/`||`
# em Spark SQL, que retorna NULL para a string inteira se qualquer campo
# envolvido for nulo — exatamente o risco citado na documentação do projeto de "filmes
# inteiros desaparecerem silenciosamente do Vector Search"

# DECISÃO 2 (agregação de atores/diretores via `collect_set` + `concat_ws`):
# usamos `collect_set` (não `collect_list`) para eliminar duplicatas de
# nomes antes de virar string — relevante porque a bridge pode ter mais de
# uma associação ator↔filme se a origem tiver ruído residual. `concat_ws`
# ignora nulos dentro da lista automaticamente, então nenhum elemento vazio
# vira "None" literal na string final
#
# DECISÃO 3 (left joins em cascata: dim_movies → fact → atores → diretores):
# preservamos TODOS os filmes de `dim_movies` no documento de contexto, não
# só os que têm fato/atores/diretores completos. Um documento de RAG "O
# filme X, lançado em ano não informado, ... elenco não informado ..." ainda
# é útil para uma busca semântica por título/sinopse, mesmo com metadados
# financeiros ausentes — melhor entregar contexto parcial ao time de IA do
# que excluir o filme inteiro da base vetorial

dim_mov = spark.table(f"{gold_schema}.dim_movies")
fact_mov = spark.table(f"{gold_schema}.fact_movies_performance")
bridge_peop = spark.table(f"{gold_schema}.bridge_movie_person")
dim_peop = spark.table(f"{gold_schema}.dim_people")

# 1. Agregação de atores por filme em string corrida (separados por vírgula)
df_atores_agg = (
    bridge_peop
    .join(dim_peop.filter(F.col("tipo_pessoa") == "Ator"), on="sk_person_id", how="inner")
    .groupBy("sk_movie_id")
    .agg(F.concat_ws(", ", F.collect_set("nome_pessoa")).alias("atores_str"))
)

# 2. Agregação de diretores por filme em string corrida
df_diretores_agg = (
    bridge_peop
    .join(dim_peop.filter(F.col("tipo_pessoa") == "Diretor"), on="sk_person_id", how="inner")
    .groupBy("sk_movie_id")
    .agg(F.concat_ws(", ", F.collect_set("nome_pessoa")).alias("diretores_str"))
)

# 3. Consolidação com fallbacks explícitos anti-nulo
df_contexto_prep = (
    dim_mov
    .join(fact_mov, on="sk_movie_id", how="left")
    .join(df_atores_agg, on="sk_movie_id", how="left")
    .join(df_diretores_agg, on="sk_movie_id", how="left")
    .select(
        dim_mov.id_filme.alias("movie_id"),
        dim_mov.titulo.alias("title"),
        F.coalesce(dim_mov.titulo, F.lit("Título não informado")).alias("v_titulo"),
        F.coalesce(dim_mov.ano_lancamento.cast("string"), F.lit("ano não informado")).alias("v_ano"),
        F.when(fact_mov.receita_usd.isNotNull(), F.concat(F.lit("$"), fact_mov.receita_usd.cast("string")))
         .otherwise(F.lit("valor não informado")).alias("v_receita"),
        F.when(fact_mov.orcamento_usd.isNotNull(), F.concat(F.lit("$"), fact_mov.orcamento_usd.cast("string")))
         .otherwise(F.lit("valor não informado")).alias("v_orcamento"),
        F.coalesce(F.col("atores_str"), F.lit("elenco não informado")).alias("v_atores"),
        F.coalesce(F.col("diretores_str"), F.lit("diretor não informado")).alias("v_diretor"),
        F.coalesce(dim_mov.sinopse, F.lit("Sinopse não disponível.")).alias("v_sinopse")
    )
)

# 4. Construção da frase corrida final sem risco de corrupção
doc_final = F.concat(
    F.lit("O filme "), F.col("v_titulo"),
    F.lit(", lançado no ano de "), F.col("v_ano"),
    F.lit(", faturou "), F.col("v_receita"),
    F.lit(" e teve um custo de "), F.col("v_orcamento"),
    F.lit(". Estrelado por "), F.col("v_atores"),
    F.lit(" e dirigido por "), F.col("v_diretor"),
    F.lit(", o filme possui a seguinte sinopse: "), F.col("v_sinopse")
)

df_gold_genai = (
    df_contexto_prep
    .select(
        F.col("movie_id"),
        F.col("title"),
        doc_final.alias("llm_context_document")
    )
)

(
    df_gold_genai.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.gold_genai_movies_context")
)

print(f" Tabela {gold_schema}.gold_genai_movies_context gravada ({df_gold_genai.count()} documentos para RAG).")

 Tabela workspace.gold.gold_genai_movies_context gravada (97611 documentos para RAG).


In [0]:

# 7. SMOKE TEST E VALIDAÇÃO GERAL DA CAMADA GOLD
# Mesmo raciocinio da célula de smoke test da Silver — lista hardcoded das
# 10 tabelas esperadas para falhar ruidosamente se alguma não existir,
# consistente com o padrão adotado nos 3 notebooks

tabelas_gold_esperadas = [
    "dim_movies",
    "dim_genres",
    "dim_companies",
    "dim_people",
    "dim_reviews",
    "bridge_movie_genre",
    "bridge_movie_person",
    "bridge_movie_company",
    "fact_movies_performance",
    "gold_genai_movies_context"
]

print("=" * 70)
print(f"RELATÓRIO DE AUDITORIA - CAMADA GOLD ({gold_schema.upper()})")
print("=" * 70)

for tab in tabelas_gold_esperadas:
    nome_tabela = f"{gold_schema}.{tab}"
    try:
        linhas = spark.table(nome_tabela).count()
        print(f" {nome_tabela:<48} | Linhas: {linhas:>8} [OK]")
    except Exception as e:
        print(f" {nome_tabela:<48} | ERRO: {e}")

print("=" * 70)

# Exibição de amostra da tabela de GenAI
display(spark.table(f"{gold_schema}.gold_genai_movies_context").limit(3))

RELATÓRIO DE AUDITORIA - CAMADA GOLD (WORKSPACE.GOLD)
 workspace.gold.dim_movies                        | Linhas:    97611 [OK]
 workspace.gold.dim_genres                        | Linhas:       19 [OK]
 workspace.gold.dim_companies                     | Linhas:    46538 [OK]
 workspace.gold.dim_people                        | Linhas:   420012 [OK]
 workspace.gold.dim_reviews                       | Linhas:    26038 [OK]
 workspace.gold.bridge_movie_genre                | Linhas:   139862 [OK]
 workspace.gold.bridge_movie_person               | Linhas:   782048 [OK]
 workspace.gold.bridge_movie_company              | Linhas:   119551 [OK]
 workspace.gold.fact_movies_performance           | Linhas:    96261 [OK]
 workspace.gold.gold_genai_movies_context         | Linhas:    97611 [OK]


movie_id,title,llm_context_document
1000004,Purple Beatz,"O filme Purple Beatz, lançado no ano de 2022, faturou valor não informado e teve um custo de valor não informado. Estrelado por Tedroy Newell, Aron Von Andrian, Izzy Jones, Steven Michael-o’hara, Erika Alexander e dirigido por Lola Atkins, o filme possui a seguinte sinopse: Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry."
1000005,Aisha Brown: The First Black Woman Ever,"O filme Aisha Brown: The First Black Woman Ever, lançado no ano de 2020, faturou valor não informado e teve um custo de valor não informado. Estrelado por Aisha Brown e dirigido por Mathieu Baer, o filme possui a seguinte sinopse: No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs."
1000007,KYLE BROWNRIGG: INTRODUCING LYLE,"O filme KYLE BROWNRIGG: INTRODUCING LYLE, lançado no ano de 2022, faturou valor não informado e teve um custo de valor não informado. Estrelado por Kyle Brownrigg e dirigido por Mathieu Baer, o filme possui a seguinte sinopse: Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle."


In [0]:

# PERGUNTA 1: RECEITA TOTAL SOMADA DE TODOS OS FILMES (EM R$)

# DECISÃO: expomos duas colunas (`receita_total_brl` decimal puro e
# `receita_total_brl_formatada` como string com separador de milhar) em vez
# de só uma. A coluna decimal serve para quem for consumir esse resultado
# programaticamente depois (ex.: um dashboard); a formatada serve para
# leitura humana direta no notebook. Ver seção "Agregando Valor" abaixo para
# uma proposta de simplificar isso em uma única saída mais clara

df_p1 = (
    spark.table(f"{gold_schema}.fact_movies_performance")
    .select(
        F.sum("receita_brl").cast("decimal(20,2)").alias("receita_total_brl"),
        F.format_number(F.sum("receita_brl"), 2).alias("receita_total_brl_formatada")
    )
)

print("--- Resposta 1: Receita Total em Reais (BRL) ---")
display(df_p1)

--- Resposta 1: Receita Total em Reais (BRL) ---


receita_total_brl,receita_total_brl_formatada
864740210242.88,"864,740,210,242.88"


In [0]:
# PERGUNTA 2: TOP 5 FILMES COM MAIOR POPULARIDADE

#Decisão:
# Cruza fact_movies_performance com dim_movies para obter o título.
# Ordena de forma decrescente pela métrica de popularidade e extrai os 5 primeiros


df_fact = spark.table(f"{gold_schema}.fact_movies_performance")
df_movies = spark.table(f"{gold_schema}.dim_movies")

df_p2 = (
    df_fact
    .join(df_movies, on="sk_movie_id", how="inner")
    .filter(F.col("popularidade").isNotNull())
    .select(
        df_movies.titulo,
        df_fact.popularidade
    )
    .orderBy(F.col("popularidade").desc())
    .limit(5)
)

print("--- Resposta 2: Top 5 Filmes por Popularidade ---")
display(df_p2)

--- Resposta 2: Top 5 Filmes por Popularidade ---


titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
The Nun II,1692.778
Meg 2: The Trench,1567.273
retribution,1547.22


Databricks visualization. Run in Databricks to view.

In [0]:

# PERGUNTA 3: VOLUME DE FILMES POR GÉNERO (DO MAIOR PARA O MENOR)

# DECISÃO: `countDistinct(sk_movie_id)` (não `count("*")`) ao agrupar por
# gênero — importante porque, em tese, um filme poderia ter um mesmo gênero
# duplicado na bridge antes do `.distinct()` daquela tabela; contar
# distinct aqui é uma segunda camada de proteção contra dupla contagem,
# redundante com o `.distinct()` já aplicado na bridge.


df_bridge_genre = spark.table(f"{gold_schema}.bridge_movie_genre")
df_genres = spark.table(f"{gold_schema}.dim_genres")

df_p3 = (
    df_bridge_genre
    .join(df_genres, on="sk_genre_id", how="inner")
    .groupBy(df_genres.nome_genero)
    .agg(F.countDistinct("sk_movie_id").alias("qtd_filmes"))
    .orderBy(F.col("qtd_filmes").desc(), F.col("nome_genero").asc())
)

print("--- Resposta 3: Contagem de Filmes por Género ---")
display(df_p3)

--- Resposta 3: Contagem de Filmes por Género ---


nome_genero,qtd_filmes
Drama,32127
Documentary,18928
Comedy,18537
Thriller,10242
Horror,9674
Romance,7619
Action,6039
Crime,4723
Animation,4454
TV Movie,4066


Databricks visualization. Run in Databricks to view.

In [0]:

# PERGUNTA 4: TOP 10 FILMES DE MAIOR RECEITA COM RANK()

#DECISÃO: `RANK()` (não `ROW_NUMBER()` nem `DENSE_RANK()`) por exigência
# explícita do projeto.
# DECISÃO 2: filtro `receita_usd > 0` antes de rankear, não só `IS NOT NULL` —
# evita que um valor residual igual a zero (que a Silver já deveria ter
# nulificado, mas é uma proteção extra em profundidade) apareça artificialmente
# no topo/fundo do ranking

from pyspark.sql.window import Window

janela_rank_receita = Window.orderBy(F.col("receita_usd").desc())

df_p4 = (
    df_fact
    .join(df_movies, on="sk_movie_id", how="inner")
    .filter(F.col("receita_usd").isNotNull() & (F.col("receita_usd") > 0))
    .withColumn("posicao_ranking", F.rank().over(janela_rank_receita))
    .select(
        "posicao_ranking",
        df_movies.titulo,
        df_fact.receita_usd,
        df_fact.receita_brl
    )
    .filter(F.col("posicao_ranking") <= 10)
    .orderBy("posicao_ranking")
)

print("--- Resposta 4: Top 10 Maiores Bilheteiras com RANK() ---")
display(df_p4)

--- Resposta 4: Top 10 Maiores Bilheteiras com RANK() ---


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


posicao_ranking,titulo,receita_usd,receita_brl
1,Avengers: Endgame,2800000000.00,14439320000.00
2,Avatar: The Way of Water,2320250281.00,11965298674.09
3,AVENGERS: INFINITY WAR,2052415039.00,10584099114.62
4,spider-man: no way home,1921847111.00,9910773366.72
5,The Lion King,1663075401.00,8576313535.42
6,Top Gun: Maverick,1488732821.00,7677246284.61
7,Barbie,1428545028.00,7366863854.89
8,The Super Mario Bros. Movie,1355725263.00,6991339608.76
9,Black Panther,1349926083.00,6961433817.42
10,Star Wars: The Last Jedi,1332698830.00,6872594596.43


In [0]:
# PERGUNTA 5: ATOR COM MAIOR QUANTIDADE DE PARTICIPAÇÕES NOS ÚLTIMOS 2 ANOS

# DECISÃO: a data-limite superior é `MAX(data_lancamento)` da própria base,
# não `current_date()`. Isso segue literalmente a regra de negócio do projeto
# importante porque a base pode não ter dados até a
# data de execução do notebook, e usar `current_date()` geraria uma janela
# de "últimos 2 anos" majoritariamente vazia se a base estiver desatualizada.

# DECISÃO 2: `countDistinct(sk_movie_id)` ao contar participações, não
# `count("*")` — protege contra o cenário em que a bridge tenha mais de uma
# linha ator↔filme (ex.: erro de dado upstream), o que inflaria a contagem
# de "participações" sem representar filmes distintos de fato.

# Ponto de atenção: usamos `.limit(1)` após `orderBy(desc)`, o que retorna
# arbitrariamente UM ator em caso de empate no topo

hoje_ref = F.current_date()

# 1. Identifica a data máxima válida realizada na base
data_maxima_base = (
    df_movies
    .filter((F.col("data_lancamento").isNotNull()) & (F.col("data_lancamento") <= hoje_ref))
    .select(F.max("data_lancamento"))
    .first()[0]
)

print(f"Data máxima de lançamento considerada na base: {data_maxima_base}")

# 2. Define o limite inferior de 2 anos
df_filmes_2_anos = (
    df_movies
    .filter(
        (F.col("data_lancamento").isNotNull()) &
        (F.col("data_lancamento") <= F.lit(data_maxima_base)) &
        (F.col("data_lancamento") >= F.add_months(F.lit(data_maxima_base), -24))
    )
    .select("sk_movie_id")
)

# 3. Cruza com as participações de atores
df_bridge_person = spark.table(f"{gold_schema}.bridge_movie_person")
df_people = spark.table(f"{gold_schema}.dim_people").filter(F.col("tipo_pessoa") == "Ator")

df_p5 = (
    df_filmes_2_anos
    .join(df_bridge_person, on="sk_movie_id", how="inner")
    .join(df_people, on="sk_person_id", how="inner")
    .groupBy(df_people.nome_pessoa)
    .agg(F.countDistinct(df_filmes_2_anos.sk_movie_id).alias("qtd_participacoes"))
    .orderBy(F.col("qtd_participacoes").desc(), F.col("nome_pessoa").asc())
    .limit(1)
)

print("--- Resposta 5: Ator com Maior Número de Filmes nos Últimos 2 Anos ---")
display(df_p5)

Data máxima de lançamento considerada na base: 2026-02-19
--- Resposta 5: Ator com Maior Número de Filmes nos Últimos 2 Anos ---


nome_pessoa,qtd_participacoes
Kevin Hart,62


In [0]:

# PERGUNTA 6: PRODUTORA COM MAIOR LUCRO NOS ÚLTIMOS 5 ANOS

# DECISÃO: somamos `lucro_usd` (não `receita_usd - orcamento_usd`
# recalculado aqui) porque o lucro já foi calculado e validado contra nulos
# na Silver. Reusar a coluna derivada evita duplicar a
# lógica de "o que fazer quando orçamento ou receita é nulo" numa segunda
# camada do pipeline.

# Mesma limitação de empate do `.limit(1)` da Pergunta 5 se aplica aqui.
# Além disso, ver na seção seguinte uma sugestão de métrica complementar
# (lucro médio por filme) para não enviesar o resultado a favor de
# produtoras com poucos filmes mas um único blockbuster

# 1. Filtra filmes dos últimos 5 anos com métricas de lucro
df_filmes_5_anos = (
    df_movies
    .filter(
        (F.col("data_lancamento").isNotNull()) &
        (F.col("data_lancamento") <= F.lit(data_maxima_base)) &
        (F.col("data_lancamento") >= F.add_months(F.lit(data_maxima_base), -60))
    )
    .select("sk_movie_id")
)

df_bridge_company = spark.table(f"{gold_schema}.bridge_movie_company")
df_companies = spark.table(f"{gold_schema}.dim_companies")

df_p6 = (
    df_filmes_5_anos
    .join(df_fact, on="sk_movie_id", how="inner")
    .filter(F.col("lucro_usd").isNotNull())
    .join(df_bridge_company, on="sk_movie_id", how="inner")
    .join(df_companies, on="sk_company_id", how="inner")
    .groupBy(df_companies.nome_produtora)
    .agg(
        F.sum("lucro_usd").cast("decimal(20,2)").alias("lucro_total_usd"),
        F.sum("lucro_brl").cast("decimal(20,2)").alias("lucro_total_brl")
    )
    .orderBy(F.col("lucro_total_usd").desc())
    .limit(1)
)

print("--- Resposta 6: Produtora Mais Lucrativa nos Últimos 5 Anos ---")
display(df_p6)

--- Resposta 6: Produtora Mais Lucrativa nos Últimos 5 Anos ---


nome_produtora,lucro_total_usd,lucro_total_brl
Universal Pictures,5790316738.00,29860084386.19
